In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

In [ ]:
f(3)

In [ ]:
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.plot(xs, ys)

In [ ]:
# производная
h = 0.00000001
x = 3
(f(x + h) - f(x)) / h

In [ ]:
a = 2
b = -3
c = 10
d = a * b + c
d

In [ ]:
h = 0.00001

a = 2
b = -3
c = 10
d1 = a * b + c
c += h
d2 = a * b + c
print(d1)
print(d2)
print('slope ', (d2 - d1) / h)

In [ ]:
from micrograd.engine import Value
from graphviz import Digraph

In [ ]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir}) #, node_attr={'rankdir': 'TB'})
    
    for n in nodes:
        dot.node(name=str(id(n)), label = "{ data %.4f | grad %.4f }" % (n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot

In [ ]:
# a very simple example
x = Value(1.0)
y = (x * 2 + 1).relu()
y.backward()
draw_dot(y)

In [ ]:
# a simple 2D neuron
import random
from micrograd import nn

random.seed(1337)
n = nn.Neuron(2)
x = [Value(1.0), Value(-2.0)]
y = n(x)
y.backward()

dot = draw_dot(y)
dot

In [ ]:
dot.render('gout')

In [ ]:
from micrograd import nn

In [860]:
x = [2.0, 3.0, -1.0]
n = nn.MLP(3, [4, 4, 1])
n(x)
# draw_dot(n(x))

Value(data=0.0, grad=0)

In [807]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

In [943]:
for k in range(20):
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()
    for p in n.parameters():
        p.data += -0.05 * p.grad
    print(k, loss.data)

0 2.713266964162978e-19
1 2.6622744356650923e-19
2 2.6122406592494295e-19
3 2.5631432787522566e-19
4 2.5149750637962237e-19
5 2.4677188788426156e-19
6 2.42134238352469e-19
7 2.3758406160486086e-19
8 2.3311989781811836e-19
9 2.287389334707411e-19
10 2.2444030512545304e-19
11 2.2022231297743936e-19
12 2.16083949656501e-19
13 2.1202350037609733e-19
14 2.0803886695273675e-19
15 2.041290039638381e-19
16 2.0029256821519853e-19
17 1.965283771577783e-19
18 1.928348724548404e-19
19 1.8921044323091995e-19


In [944]:
ypred = [n(x) for x in xs]
ypred

[Value(data=0.9999999999711503, grad=0),
 Value(data=-1.0000000004172946, grad=0),
 Value(data=-1.0000000000992724, grad=0),
 Value(data=0.9999999999711503, grad=0)]